# Ideas planner, second list, idea 1: count LoFTR matches against the coarse alignment, not a blind RANSAC
The coarse peak already fixes the shift to about one cell. So at a candidate place the canvas and the map tile are already aligned, and a right place should give many matches whose two points lie almost at the same pixel; a false place gives matches that are scattered. Lower the match confidence (0.22 -> 0.05..0.22), count the matches that sit within T metres of the same pixel (T = 4 / 8 / 12 / 20 m, converted to pixels with the ground size of the level) and subtract the count a random scatter would give (chance = all matches x tolerance area / canvas area). Reading: the excess over chance (as a z-like number) of the TRUE place against every false candidate.
Data: official150 dump. Skipped photos at <= 150 m (the 12 where ours loses marks, truth injected as a candidate) and controls (answered right). The false candidates are the 10 fine-matched candidates of each standard query that lie more than 4 sigma from the truth. Six standard queries per photo.
Falsification (planned before running): dead if the truth does not beat every false candidate on at least 8 of the 12 skipped photos at some (confidence, tolerance). The grid has 16 cells, so one lucky cell is not enough: look for a region of the grid, and the controls must also win.

In [1]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

official official150 pool 500 limit 0


In [2]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, level_gsd

def make_pipeline(headings, scales, confidence=0.22, coarse_threshold=0.21):
    pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=confidence, satellite="new",
                                map_margin_m=None, loftr_coarse_threshold=coarse_threshold, dino_weight=None, headings=headings, scale_factors=scales)
    start = time.time()
    pipeline.setup(POOL_PHOTOS, 7)
    print("setup %.0f s" % (time.time() - start))
    return pipeline

STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
dump = replay.load_set(T7 / "rich_dump" / TAG)

In [3]:
pipeline = make_pipeline([-5.0, 0.0, 5.0], [1.0, 1.05], confidence=0.05, coarse_threshold=0.05)
from fine_matching.loftr_matcher import candidate_matches, stage2_from_matches, reliable_mask
LOW = 150.0
low, control = [], []
for pid, photo in dump.items():
    if photo["height"] > LOW:
        continue
    result = replay.decide_current(photo)
    if result["stage"] == "skip":
        low.append(pid)
    elif result["err_sigma"] <= 1.0:
        control.append(pid)
control = control[:int(os.environ.get("CONTROL_N", "16"))]
if N_LIMIT:
    low, control = low[:N_LIMIT], control[:N_LIMIT]
print(len(low), "skipped at <= 150 m;", len(control), "controls")

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 112 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 126 s
12 skipped at <= 150 m; 16 controls


In [4]:
C_LIST = [0.05, 0.10, 0.15, 0.22]
T_LIST = [4.0, 8.0, 12.0, 20.0]
SIZE = 224

def aligned_features(stored, k, level):
    item = stored["items"][k]
    if item is None:
        return None
    reliable = reliable_mask(stored["inside"], stored["border_px"])
    a, b, confidence = item["points_a"], item["points_b"], item["confidence"]
    columns = np.clip(np.round(a[:, 0]).astype(int), 0, reliable.shape[1] - 1)
    rows_ = np.clip(np.round(a[:, 1]).astype(int), 0, reliable.shape[0] - 1)
    usable = reliable[rows_, columns]
    gsd = level_gsd(level)
    gap_m = np.hypot(b[:, 0] - a[:, 0], b[:, 1] - a[:, 1]) * gsd
    out = {}
    for c in C_LIST:
        chosen = usable & (confidence >= c)
        everything = int(chosen.sum())
        for t in T_LIST:
            near = int((chosen & (gap_m <= t)).sum())
            chance = everything * min(1.0, np.pi * (t / gsd) ** 2 / (SIZE * SIZE))
            out["al_C%.2f_T%d" % (c, t)] = near
            out["z_C%.2f_T%d" % (c, t)] = (near - chance) / np.sqrt(chance + 1.0)
    return out

rows = []
start = time.time()
for group, ids in [("skipped", low), ("control", control)]:
    for pid in ids:
        photo = dump[pid]; item = pipeline.test_set.items[pid]; level = photo["level"]
        truth = [float(v) for v in photo["truth"]]; sigma = photo["sigma"]
        for q in replay.query_ids(photo, STANDARD):
            query = photo["queries"][q]
            false_peaks = [p for p in query["peaks"][:10] if p[1] < 1e8 and np.hypot(p[1] - truth[0], p[2] - truth[1]) / sigma > 4.0]
            peaks = np.array([[1.0, truth[0], truth[1], 0.0, 0.0]] + [list(p) for p in false_peaks])
            canvas, valid = render_query(item, pipeline.camera, level, query_geometry(query["heading"], query["scale"]))
            stored = candidate_matches(pipeline.matcher, canvas, valid, level, peaks, pipeline.level_images)
            second = stage2_from_matches(stored, level, 0.22)
            for k in range(len(peaks)):
                features = aligned_features(stored, k, level)
                if features is None:
                    continue
                rows.append(dict(group=group, photo=pid, height=photo["height"], query=q, kind="truth" if k == 0 else "false", ransac22=float(second["inliers"][k]), **features))
        torch.cuda.empty_cache(); gc.collect()
    print(group, "done %.0f s" % (time.time() - start), flush=True)
A = pd.DataFrame(rows)
A.to_csv(T7 / ("idea1b_coarse_aligned_%s.csv" % os.environ.get("OUT_TAG", "official")), index=False)
print(len(A), "candidate rows")

skipped done 30 s


control done 69 s


1757 candidate rows


In [5]:
from sklearn.metrics import roc_auc_score
features = ["ransac22"] + [c for c in A.columns if c.startswith("z_C")]
best = A.groupby(["group", "photo", "kind"])[features].max().reset_index()
table = []
for f in features:
    wide = best.pivot(index=["group", "photo"], columns="kind", values=f).dropna()
    wins = (wide["truth"] > wide["false"]).groupby(level="group").sum()
    counts = (wide["truth"] > wide["false"]).groupby(level="group").count()
    row = dict(feature=f, skipped_wins="%d/%d" % (wins.get("skipped", 0), counts.get("skipped", 0)), control_wins="%d/%d" % (wins.get("control", 0), counts.get("control", 0)))
    labels = (A.kind == "truth").astype(int)
    row["auc_candidates"] = round(roc_auc_score(labels, A[f]), 3) if 0 < labels.sum() < len(labels) else np.nan
    table.append(row)
R = pd.DataFrame(table)
R.to_csv(T7 / ("idea1b_summary_%s.csv" % os.environ.get("OUT_TAG", "official")), index=False)
print(R.to_string(index=False))

    feature skipped_wins control_wins  auc_candidates
   ransac22         5/12        16/16           0.764
 z_C0.05_T4        11/12        14/16           0.834
 z_C0.05_T8        11/12        16/16           0.909
z_C0.05_T12        10/12        16/16           0.945
z_C0.05_T20        10/12        16/16           0.963
 z_C0.10_T4        10/12        15/16           0.745
 z_C0.10_T8        11/12        16/16           0.879
z_C0.10_T12        11/12        16/16           0.947
z_C0.10_T20        11/12        16/16           0.957
 z_C0.15_T4        10/12        14/16           0.680
 z_C0.15_T8        11/12        16/16           0.838
z_C0.15_T12        11/12        16/16           0.923
z_C0.15_T20        10/12        16/16           0.948
 z_C0.22_T4         8/12        13/16           0.624
 z_C0.22_T8        11/12        15/16           0.781
z_C0.22_T12         8/12        15/16           0.884
z_C0.22_T20         8/12        16/16           0.925


In [6]:
wide = best.pivot(index=["group", "photo"], columns="kind", values="ransac22").dropna()
print("RANSAC inliers (today): truth best vs false best per photo")
print(wide.round(1).to_string())
choice = R.sort_values(["skipped_wins"], ascending=False).iloc[0]
print("best grid cell on the skipped photos:", choice.feature, choice.skipped_wins, "controls", choice.control_wins)

RANSAC inliers (today): truth best vs false best per photo
kind           false  truth
group   photo              
control 1        8.0   63.0
        4        0.0    7.0
        6        4.0    7.0
        8        0.0   24.0
        31       0.0    9.0
        36       0.0    2.0
        45       0.0   17.0
        55       0.0   12.0
        61       0.0   13.0
        64       6.0   11.0
        72       0.0    8.0
        96       0.0   20.0
        100      4.0  107.0
        107      5.0   31.0
        111      7.0   26.0
        128      0.0   13.0
skipped 11       0.0    0.0
        35       0.0    0.0
        66       0.0    0.0
        172      0.0    7.0
        176      0.0    2.0
        200      0.0    3.0
        217      0.0    0.0
        266      0.0    0.0
        286      0.0    5.0
        297      0.0    5.0
        306      0.0    0.0
        329      0.0    0.0
best grid cell on the skipped photos: z_C0.22_T20 8/12 controls 16/16
